In [ ]:
%pip install torch
%pip install --no-deps xformers trl peft accelerate bitsandbytes
%pip install unsloth
%pip install -U transformers 
%pip install datasets


  Using cached torch-2.7.1-cp312-none-macosx_11_0_arm64.whl.metadata (29 kB)
  Using cached setuptools-80.9.0-py3-none-any.whl.metadata (6.6 kB)
   ━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.8/68.6 MB 5.9 MB/s eta 0:00:09^C
ERROR: Operation cancelled by user
Note: you may need to restart the kernel to use updated packages.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 6.1 MB/s eta 0:00:00a 0:00:01
  Installing build dependencies ... |

In [ ]:
from unsloth import FastLanguageModel 
import torch
load_4_bit = True
device_map = "auto"
max_sequence = 2048

Quantized_model = "unsloth/mistral-7b-instruct-v0.2-bnb-4bit"
model, Tokenizer = FastLanguageModel.from_pretrained(
    Quantized_model,
    max_seq_length=max_sequence,
    device_map=device_map,
    load_in_4bit=load_4_bit,
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16, 
    lora_alpha = 16, 
    lora_dropout=0,
    bias=None,
    use_gradient_checkingpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None
    )
    

In [ ]:
from google.colab import drive
from unsloth.chat_templates import get_chat_template
from dataset import load_dataset
drive.mount("/content/drive")
path = ''

tokenizer = get_chat_template(
    tokenizer, 
    chat_template="chatml", 
    map_eos_token=True
)


def formatting_function(examples):
  convos = []
  for i in range(len(examples["Question"])):
    conversation_format = [
        {"from": "human", "value":examples["Question"][i]},
        {"from":"Gpt", "value":examples["Answer"][i]}
    ]
    convos.append(conversation_format)
  texts = [tokenizer.apply_chat_templates(convo, tokenize=False) for convo in convos]
  return {"Texts": texts}

from datasets import load_datasets

In [ ]:
from unsloth.chat_templates import get_chat_template
from datasets import load_dataset
path = '/cleaned_data.json'

tokenizer = get_chat_template(
    tokenizer, 
    chat_template="chatml", 
    map_eos_token=True
)


def formatting_function(examples):
  convos = []
  for i in range(len(examples["Question"])):
    conversation_format = [
        {"from": "human", "value":examples["Question"][i]},
        {"from":"Gpt", "value":examples["Answer"][i]}
    ]
    convos.append(conversation_format)
  texts = [tokenizer.apply_chat_templates(convo, tokenize=False) for convo in convos]
  return {"Texts": texts}

dataset = load_dataset("json", data_files="/cleaned_data.json", split="train")
dataset = dataset.map(formatting_function, batched=True)



In [ ]:
from trl import SFTConfig, SFTTrainer
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_sequence,
    packing = False, # Can make training 5x faster for short sequences.
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # 
    ),
)

In [ ]:
#Finally training, woohoo!
trainer_stats = trainer.train()